# Experiment 3: Embedding Vectorizers with Logistic Regression

Comparing TF-IDF against Transformer embeddings (DistilBERT) for sentiment classification using Logistic Regression.

In [ ]:
import os
import re
import string
import warnings
import pandas as pd
import numpy as np
import scipy.sparse
import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import mlflow
import mlflow.sklearn
import dagshub

warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore")

## 1. Setup & Data Loading

In [ ]:
DATA_PATH = '../data/balanced_sentiment_dataset.csv' if os.path.exists('../data/balanced_sentiment_dataset.csv') else 'data/balanced_sentiment_dataset.csv'

CONFIG = {
    "data_path": DATA_PATH,
    "test_size": 0.2,
    "experiment_name": "Embedding Vectorizers with Logistic Regression"
}

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = ''.join([char for char in text if not char.isdigit()])
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return " ".join(words)

df = pd.read_csv(CONFIG["data_path"])
df['text'] = df['text'].apply(preprocess_text)
print(f"Loaded and preprocessed {len(df)} samples.")

## 2. Transformer Feature Extraction (DistilBERT)

In [ ]:
print("Loading DistilBERT tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
transformer_model = AutoModel.from_pretrained('distilbert-base-uncased')
transformer_model.eval()

def extract_transformer_embeddings(texts, batch_size=32):
    vectors = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')
            outputs = transformer_model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            vectors.append(cls_embeddings)
    return np.vstack(vectors)

## 3. Model Training & Comparison

In [ ]:
# Run comparison on sample subset
sample_df = df.sample(n=min(len(df), 5000), random_state=42)
X_texts = sample_df['text'].tolist()
y = sample_df['sentiment'].values

# 1. TF-IDF
tfidf_vec = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vec.fit_transform(X_texts)
X_tr, X_te, y_tr, y_te = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

clf_tfidf = LogisticRegression(max_iter=1000)
clf_tfidf.fit(X_tr, y_tr)
y_pred_tfidf = clf_tfidf.predict(X_te)
print(f"TF-IDF + LogisticRegression Accuracy: {accuracy_score(y_te, y_pred_tfidf):.4f}")

# 2. Transformer Embeddings
X_trans = extract_transformer_embeddings(X_texts)
X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(X_trans, y, test_size=0.2, random_state=42)

clf_trans = LogisticRegression(max_iter=1000)
clf_trans.fit(X_tr_t, y_tr_t)
y_pred_trans = clf_trans.predict(X_te_t)
print(f"DistilBERT + LogisticRegression Accuracy: {accuracy_score(y_te_t, y_pred_trans):.4f}")